# Introduction

# Stage 07 — Synthesis and reproducibility

**Pipeline position:** last. Reads everything the pipeline produced; writes the manifest and states
the verdict.

## What this stage does

Four checks and a synthesis. The checks are what let a reader trust the synthesis.

1. **Reconstruct every published number from the exported rows.** Re-read
   `artifacts/player_season_results.csv` from disk — not any in-memory frame — and independently
   re-derive every Sleeper-ADP cell of `artifacts/threshold_summary.csv`. This proves the summary was
   computed on the frame that was exported, that the split labels mean what they claim, and that a
   reader holding only the CSVs can rebuild every number without running anything.
2. **Independently re-derive every rank**, by a separately written expression, and confirm the gap
   identities hold. This is the check that catches the most damaging silent error available here — a
   rank taken over the wrong grouping.
3. **Re-hash every input**, catching an analysis that rewrote the data it was measuring, and verify
   the archived originals against the digests captured when they were moved.
4. **Audit all eight notebooks structurally** — Introduction first, Conclusion last, every code cell
   sandwiched between `### Explain` and `### Interpretation`, no adjacent code cells, execution counts
   present, no error outputs, no prospective placeholder language, and no `.py` analysis script
   anywhere in the project.

Then the manifest is written and the verdict stated.

## A note on the two-pass execution

This notebook audits notebooks, including itself. The pipeline is therefore executed **twice**: on the
second pass every file on disk is already fully executed, so the execution-count and stored-output
checks can be decided about real artifacts rather than about intent. A `PENDING` result on those lines
means a first pass; the shipped notebooks show the second-pass result.

## Inputs and outputs

| Direction | Path |
|---|---|
| in | every artifact, every interim file, all eight notebooks, all nine production inputs |
| out | `artifacts/manifest.json` |

### Explain — load the shared library

Every stage notebook begins here. It loads `00_shared_pipeline.ipynb` using the repo's convention
(`memory/prefer-ipynb-not-py.md`, mirroring the loader in `betting/predict_totals.ipynb` cell 4):
**json + exec over the library's code cells**, never `%run` (brittle across nbclient / papermill /
VSCode) and never a `.py` module (the repo is notebook-centric by rule).

`RUN_TESTS = False` and `SHARED_VERBOSE = False` are set **before** the exec, so the library's inline
tests are skipped and its configuration banner stays silent — those belong to a standalone run of the
library, not to every consumer.

The cell prints a compact load record: the SHA-256 of the library notebook itself, the count of names
imported, and the pinned parameters. Recording the library's hash means each stage's output states
exactly which version of the shared code produced it — if the library changes, the stages' recorded
hashes diverge and the mismatch is visible rather than silent.

In [1]:
import json as _json
from pathlib import Path as _Path


def _exec_notebook(path, glob):
    """Execute every code cell of a notebook into `glob` (repo convention: json + exec)."""
    with open(path, encoding="utf-8") as _fh:
        _nb = _json.load(_fh)
    for _cell in _nb["cells"]:
        if _cell["cell_type"] == "code":
            exec("".join(_cell["source"]), glob)


RUN_TESTS = False          # skip the library's inline self-tests in a consumer
SHARED_VERBOSE = False     # suppress the library's configuration banner
_SHARED = "00_shared_pipeline.ipynb"
_before = set(globals())
_exec_notebook(_SHARED, globals())
_loaded = sorted(n for n in set(globals()) - _before
                 if not n.startswith("_") and n not in {"RUN_TESTS", "SHARED_VERBOSE"})

print(f"loaded {_SHARED}")
print(f"  library sha256 : {sha256_file(_SHARED)}")
print(f"  names imported : {len(_loaded)}")
print(f"  functions      : {[n for n in _loaded if callable(globals()[n])]}")
print(f"  repo           : {REPO.name}   project: {PROJECT.name}")
print(f"  seasons {TEST_SEASONS} | thresholds {THRESHOLDS} | populations {list(POPULATIONS)}")
print(f"  seed {SEED} | perms {N_PERM:,} | boots {N_BOOT:,}")

loaded 00_shared_pipeline.ipynb
  library sha256 : d3e28a60fab75caf19c5387de293057de842c21c3c088edbc204acd89972b469
  names imported : 48
  functions      : ['Path', 'add_signals', 'boot_index_matrix', 'bootstrap_lift', 'build_ranks', 'canonical_strata', 'correct_vec', 'datetime', 'logistic_design', 'logistic_newton', 'norm', 'perm_sign_matrix', 'permutation_test', 'population_slice', 'sha256_file', 'spearmanr', 'summarise_cell', 'thr_col', 'timezone', 'wilson']
  repo           : JoSchoAnalytics   project: adp_consensus_agreement_2026-08-02
  seasons [2021, 2022, 2023, 2024, 2025] | thresholds [0.0, 5.0, 7.5, 10.0] | populations ['all_adp', 'drafted_top180']
  seed 20260802 | perms 10,000 | boots 10,000


### Interpretation — library loaded, this stage is anchored to it

The load record confirms the shared library executed cleanly and lists the names now in scope,
including the analysis functions this stage calls. The pinned parameters match the study's
declaration — seasons 2021–2025, thresholds `[0, 5, 7.5, 10]`, both populations, seed 20260802 — so
this notebook cannot silently disagree with its siblings about what a rank is or how a hit rate is
scored.

The **library SHA-256 is printed and recorded**. Every stage prints the same digest, which is what
makes "all seven stages ran against the same library" a checkable claim rather than an assumption;
stage 07 re-hashes the library and compares.

`RUN_TESTS=False` means the library's self-tests did not run here — they belong to a standalone run
of `00_shared_pipeline.ipynb`, which is the gate for this pipeline being trustworthy at all.

**This stage reads:** every artifact, every interim file, all eight notebooks, and all nine production inputs
**and writes:** `artifacts/manifest.json`

### Explain — reconstruct every published number from the exported rows

The first and most important reproducibility guarantee.

**What it does.** Re-reads `artifacts/player_season_results.csv` **from disk** — not any in-memory
frame — and for **every** Sleeper-ADP row of `artifacts/threshold_summary.csv` independently
re-derives the population, universe, panel, threshold and split filters, then recomputes `n`, `hits`,
`misses` and `ties`. Any disagreement fails the notebook.

This is a stronger check than it first appears. It proves three things at once: the summary was
computed on the frame that was actually exported; the split labels mean what they claim; and a reader
holding only the CSVs can rebuild every published number without running any of these notebooks.

**Underdog rows are excluded, for a stated reason.** They are computed against a different market
whose per-player rows are not part of this export. That exclusion is reported in the output rather
than silently applied.

**Second check, in the same cell.** Every rank in the export is re-derived by a separately written
expression, and the three gap identities (`adp_rank - X_rank`) are verified. This is what catches the
most damaging silent error available in this study — a rank taken over the wrong grouping, which
would still produce a plausible-looking column.

In [2]:
_rl = pd.read_csv(ARTIFACTS / "player_season_results.csv")
_sm = pd.read_csv(ARTIFACTS / "threshold_summary.csv")
_sleeper_rows = _sm[_sm.market == "sleeper_adp"]
print(f"row-level file : {len(_rl):,} rows")
print(f"summary file   : {len(_sm):,} rows ({len(_sleeper_rows):,} sleeper_adp, "
      f"{len(_sm) - len(_sleeper_rows)} underdog)\n")

print("RECONSTRUCTION CHECK — rebuild every Sleeper-ADP summary cell from the exported row-level CSV")
print("=" * 100)
_SPLIT_COL = {"direction": "direction", "position": "position", "season": "season", "group": "group"}
_checked = _failed = 0
_failures = []
for _, r in _sleeper_rows.iterrows():
    d = _rl[(_rl.population == r.population) & (_rl.universe == r.universe)
            & _rl.complete & _rl.season.isin(PANELS[r.panel])]
    cell = d[d[thr_col(r.threshold)]]
    if r.split != "all":
        cell = cell[cell[_SPLIT_COL[r.split]].astype(str) == str(r.split_value)]
    got = (len(cell), int((cell.outcome == "hit").sum()),
           int((cell.outcome == "miss").sum()), int((cell.outcome == "tie").sum()))
    want = (int(r.n), int(r.hits), int(r.misses), int(r.ties))
    _checked += 1
    if got != want:
        _failed += 1
        _failures.append((r.population, r.universe, r.panel, r.threshold, r.split, r.split_value,
                          want, got))
print(f"  summary cells checked  : {_checked:,}")
print(f"  reconstruction failures: {_failed}")
for f in _failures[:10]:
    print("   ", f)
assert _failed == 0, f"{_failed} summary cells could not be reconstructed"
print("  => every published Sleeper-ADP number is rebuildable from the exported CSVs alone.")

print("\nINDEPENDENT RANK RE-DERIVATION (recomputed within population/universe/season/position)")
print("=" * 100)
_g = _rl.groupby(["population", "universe", "season", "position"])
_re = pd.DataFrame({"adp_rank": _g["adp_half_ppr"].rank(method="min", ascending=True),
                    "model_rank": _g["model_pred"].rank(method="min", ascending=False),
                    "sleeper_rank": _g["sleeper"].rank(method="min", ascending=False),
                    "actual_rank": _g["actual_half_ppr"].rank(method="min", ascending=False)})
_mismatch = {c: int((~np.isclose(_re[c], _rl[c], equal_nan=True)).sum()) for c in _re.columns}
for c, n in _mismatch.items():
    print(f"  {c:14s} mismatching rows: {n}")
assert sum(_mismatch.values()) == 0, f"rank re-derivation disagrees: {_mismatch}"
_gap_bad = int(sum((~np.isclose(_rl.adp_rank - _rl[a], _rl[b], equal_nan=True)).sum()
                   for a, b in [("model_rank", "model_gap"), ("sleeper_rank", "sleeper_gap"),
                                ("actual_rank", "actual_gap")]))
print(f"  gap identities (adp_rank - X_rank) mismatching rows: {_gap_bad}")
assert _gap_bad == 0

row-level file : 5,315 rows
summary file   : 1,346 rows (1,298 sleeper_adp, 48 underdog)

RECONSTRUCTION CHECK — rebuild every Sleeper-ADP summary cell from the exported row-level CSV


  summary cells checked  : 1,298
  reconstruction failures: 0
  => every published Sleeper-ADP number is rebuildable from the exported CSVs alone.

INDEPENDENT RANK RE-DERIVATION (recomputed within population/universe/season/position)
  adp_rank       mismatching rows: 0
  model_rank     mismatching rows: 0
  sleeper_rank   mismatching rows: 0
  actual_rank    mismatching rows: 0
  gap identities (adp_rank - X_rank) mismatching rows: 0


### Interpretation — every published number is rebuildable, and the ranks survive re-derivation

**All 1,298 Sleeper-ADP summary cells reconstructed with 0 failures**, from a file re-read off disk
rather than from anything still in memory. That proves the summary was computed on the frame that was
exported, that the split labels mean what they claim, and that a reader holding only
`artifacts/player_season_results.csv` and `artifacts/threshold_summary.csv` can rebuild every
published number without running a single notebook.

The 48 Underdog rows are excluded for the stated reason — a different market whose per-player rows are
not in this export — and the exclusion is printed rather than quietly applied.

**The rank re-derivation returned 0 mismatching rows on all four ranks and 0 on all three gap
identities**, computed by a separately written expression rather than reused from stage 02. This is
the check that would catch the single most damaging silent error available here: a rank taken over the
wrong grouping, which would still look like a perfectly ordinary rank column while comparing
quarterbacks to tight ends. It did not happen.

Together these two checks mean the exported artifacts are internally consistent and independently
verifiable. Next: confirm nothing on disk moved while all this was being computed.

### Explain — re-hash every input and verify the archived originals

Two provenance guarantees closing the loop.

**1. Every input must be byte-identical to its stage-01 digest.** Re-hashing at the end catches the
failure mode where an analysis silently rewrites the data it is measuring — a pipeline that
regenerates its own inputs mid-run would otherwise produce self-consistent nonsense. Stage 01 also
compared these against the archived run, so a pass here means the inputs were identical to the
original run when read *and* identical again when the pipeline finished.

**2. The eight archived originals must match the digests captured when they were moved.**
`archive/original_2026-08-02/archive_move_receipt.json` records each file's SHA-256 both immediately
before and immediately after its move. Verifying against it confirms the archive is a faithful copy
of the original single-notebook run rather than a re-derivation of it — which is what makes the
then-versus-now comparison in the next cell a real comparison.

**3. The shared library's own hash** is re-computed and compared against the digest each stage
recorded when it loaded the library, so "all eight notebooks ran against the same library" is a
checked claim rather than an assumption.

In [3]:
_diag = json.loads((INTERIM / "stage01_diagnostics.json").read_text(encoding="utf-8"))
INPUT_HASHES = _diag["input_hashes"]

print("INPUT RE-HASH — sources must be unchanged since stage 01")
print("=" * 100)
_drift = []
for label, rec in INPUT_HASHES.items():
    now = sha256_file(REPO / rec["path"])
    ok = now == rec["sha256"]
    if not ok:
        _drift.append(label)
    print(f"  {'UNCHANGED' if ok else 'DRIFTED  '}  {label:18s} {now[:32]}...")
assert not _drift, f"source artifacts changed during the run: {_drift}"

print("\nARCHIVE INTEGRITY — the original 2026-08-02 run, verified against its move receipt")
print("=" * 100)
_receipt = json.loads((ARCHIVE / "archive_move_receipt.json").read_text(encoding="utf-8"))
_bad = []
for name, rec in _receipt.items():
    now = sha256_file(ARCHIVE / name)
    ok = now == rec["sha256_after"] == rec["sha256_before"]
    if not ok:
        _bad.append(name)
    print(f"  {'OK      ' if ok else 'MISMATCH'}  {name:36s} {now[:28]}...  {rec['bytes']:>9,} B")
assert not _bad, f"archived originals altered: {_bad}"
print(f"\n  {len(_receipt)} archived files, all byte-identical to their pre-move state.")

print("\nSHARED LIBRARY — one definition across all stages")
print("=" * 100)
_lib_sha = sha256_file(PROJECT / "00_shared_pipeline.ipynb")
print(f"  00_shared_pipeline.ipynb sha256: {_lib_sha}")
_recorded = []
for nb_name in NOTEBOOKS[1:]:
    _nb = json.loads((PROJECT / nb_name).read_text(encoding="utf-8"))
    for c in _nb["cells"]:
        if c["cell_type"] == "code":
            for o in c.get("outputs", []):
                for ln in ("".join(o.get("text", ""))).splitlines():
                    if "library sha256" in ln:
                        _recorded.append((nb_name, ln.split(":")[-1].strip()))
                        break
            break
if _recorded:
    _agree = [n for n, h in _recorded if h == _lib_sha]
    print(f"  stages recording a library hash : {len(_recorded)}")
    print(f"  stages agreeing with the current: {len(_agree)}")
    for n, h in _recorded:
        print(f"    {'MATCH ' if h == _lib_sha else 'DIFFER'}  {n}")
else:
    print("  (no stage has recorded a library hash yet — first pass)")

INPUT RE-HASH — sources must be unchanged since stage 01
  UNCHANGED  walkforward_RB     c8c0e1584a18452adcb2c3510b1ff911...
  UNCHANGED  walkforward_WR     49f6b0d69f796d90c1fe5b3bd9a1fdd0...
  UNCHANGED  walkforward_TE     ab663974a8888334ee7fd7cf69c893aa...
  UNCHANGED  walkforward_QB     b2545b0c55ab3dfbd0ff378b60d60a4a...
  UNCHANGED  season_dataset     dfdf38d9372c830b9fdfffe914024000...
  UNCHANGED  builder_RB         7ffb77d4db746f3ebbc6c2ecb475481b...
  UNCHANGED  builder_WR         bd29dc1856be12d135d0e19594d9d8af...
  UNCHANGED  builder_TE         71f8278e9014de52b2d410da85b4aa97...
  UNCHANGED  builder_QB         fae87aa98030558f15441deb1a7482d3...

ARCHIVE INTEGRITY — the original 2026-08-02 run, verified against its move receipt
  OK        adp_consensus_experiment.ipynb       ac938f1466f190fbaf1ac00b5cff...     40,814 B
  OK        REPORT.md                            72e1beea03a10717a8c3e519a4d9...     16,050 B
  OK        threshold_summary.csv                6a6c81b0fd

### Interpretation — nothing moved, and the archive is faithful

**All nine inputs re-hashed UNCHANGED.** Nothing this pipeline did rewrote a source. Combined with
stage 01's match against the archived manifest, the inputs were identical to the original run when
read and identical again when the pipeline finished — the window in which a self-modifying analysis
could have hidden is closed at both ends.

**All eight archived originals verified byte-identical** to the digests captured at the moment they
were moved: the two handoff briefs, the original single-notebook experiment, its `REPORT.md`, and its
four artifacts. The archive is a faithful copy of the 2026-08-02 run rather than a re-derivation, so
the then-versus-now comparison is a real comparison rather than a tautology.

**The shared-library check** confirms every stage notebook recorded the same library SHA-256 as the
file currently on disk. That turns "all eight notebooks ran against the same definitions of
`build_ranks`, `summarise_cell` and the resamplers" from an assumption into something a reader can
verify from the stored outputs. On a first pass, before the later stages have executed, this section
reports that no hash has been recorded yet — which is honest rather than convenient.

### Explain — audit the structure of all eight notebooks

The pipeline inspecting itself. Each notebook is parsed as JSON and checked against every structural
rule from the revamp brief and `memory/prefer-ipynb-not-py.md`:

- the first cell is markdown beginning `# Introduction`;
- the last cell is markdown beginning `# Conclusion and Next Steps`;
- every code cell's immediate predecessor is markdown whose heading begins `### Explain`;
- every code cell's immediate successor is markdown whose heading begins `### Interpretation`;
- no two code cells are adjacent;
- every code cell carries an execution count, and counts run 1..N;
- no code cell contains an error output or traceback;
- every code cell has stored output;
- no Interpretation cell contains prospective placeholder language — interpretations must describe
  observed output, not what a cell is going to produce;
- no companion `.py` analysis or validation script exists anywhere in the project folder.

**Two self-referential carve-outs, both stated rather than hidden.** This cell raises on a structural
failure, so on the following pass it would detect its own traceback and fail forever; the error scan
therefore excludes **this cell only**, and reports its prior state on a separate line. Every other
cell in every notebook is scanned with no exception. Similarly, the placeholder vocabulary is
assembled from string fragments so that the check's own source line is not itself a tripwire.

**How execution-state checks are possible from inside the run.** The pipeline is executed **twice**.
On the second pass every notebook on disk is already fully executed, so these checks are decided about
real artifacts. A `PENDING` result means a first pass.

In [4]:
print("STRUCTURAL AUDIT — all eight pipeline notebooks")
print("=" * 116)
_placeholders = ["this " + "should show", "we " + "expect", "should " + "produce",
                 "PLACE" + "HOLDER", "TO" + "DO"]      # split so this line is not its own tripwire
_self_nb = "07_synthesis_and_reproducibility.ipynb"
_SELF_MARKER = "STRUCTURAL AUDIT " + "— all eight pipeline notebooks"

STRUCTURAL_AUDIT, _all_fail = {}, []
for nb_name in NOTEBOOKS:
    cells = json.loads((PROJECT / nb_name).read_text(encoding="utf-8"))["cells"]
    src = lambda c: "".join(c["source"])
    code = [c for c in cells if c["cell_type"] == "code"]
    interp = [c for c in cells if c["cell_type"] == "markdown"
              and src(c).lstrip().startswith("### Interpretation")]
    executed = bool(code) and all(c.get("execution_count") is not None for c in code)
    # identify THIS cell by a unique marker in its own source -- it is not the last code cell
    self_idx = next((i for i, c in enumerate(cells) if c["cell_type"] == "code"
                     and _SELF_MARKER in "".join(c["source"])), None) if nb_name == _self_nb else None

    bad_prev = [i for i, c in enumerate(cells) if c["cell_type"] == "code" and not
                (i > 0 and cells[i - 1]["cell_type"] == "markdown"
                 and src(cells[i - 1]).lstrip().startswith("### Explain"))]
    bad_next = [i for i, c in enumerate(cells) if c["cell_type"] == "code" and not
                (i + 1 < len(cells) and cells[i + 1]["cell_type"] == "markdown"
                 and src(cells[i + 1]).lstrip().startswith("### Interpretation"))]
    adj = [i for i, c in enumerate(cells)
           if c["cell_type"] == "code" and i > 0 and cells[i - 1]["cell_type"] == "code"]
    errs, self_err = [], False
    for i, c in enumerate(cells):
        if c["cell_type"] != "code":
            continue
        if any(o.get("output_type") == "error" or "traceback" in o for o in c.get("outputs", [])):
            if i == self_idx:
                self_err = True
            else:
                errs.append(i)
    ec = [c.get("execution_count") for c in code]
    stale = [i for i, c in enumerate(interp)
             if any(p.lower() in src(c).lower() for p in _placeholders)]

    checks = {
        "first cell '# Introduction'": cells[0]["cell_type"] == "markdown"
                                       and src(cells[0]).lstrip().startswith("# Introduction"),
        "last cell '# Conclusion and Next Steps'": cells[-1]["cell_type"] == "markdown"
                                       and src(cells[-1]).lstrip().startswith("# Conclusion and Next Steps"),
        "'### Explain' above every code cell": not bad_prev,
        "'### Interpretation' below every code cell": not bad_next,
        "no adjacent code cells": not adj,
        "no error output / traceback": not errs,
        "no prospective placeholder language": not stale,
        "every code cell has an execution count": (all(e is not None for e in ec) if executed else None),
        "execution counts sequential 1..N": (ec == list(range(1, len(code) + 1)) if executed else None),
        "every code cell has stored output": (all(c.get("outputs") for c in code) if executed else None),
    }
    if nb_name == _self_nb:
        checks["this audit cell did not fail on the previous pass"] = not self_err

    decided = [v for v in checks.values() if v is not None]
    fails = [k for k, v in checks.items() if v is False]
    _all_fail += [f"{nb_name}: {k}" for k in fails]
    STRUCTURAL_AUDIT[nb_name] = {
        "cells": len(cells), "markdown": len(cells) - len(code), "code": len(code),
        "interpretation_cells": len(interp), "executed": executed,
        "checks": {k: ("pass" if v is True else "pending" if v is None else "FAIL")
                   for k, v in checks.items()}}
    print(f"\n  {nb_name}")
    print(f"    cells {len(cells):>3} ({len(cells)-len(code):>3} md / {len(code):>2} code)  "
          f"executed={executed}  -> {sum(bool(v) for v in decided)}/{len(decided)} decided checks pass"
          + ("" if not fails else f"   FAILURES: {fails}"))

_stray = sorted(str(p.relative_to(PROJECT)) for p in PROJECT.rglob("*.py"))
print(f"\n  companion .py analysis/validation scripts anywhere in the project: {len(_stray)} {_stray}")
if _stray:
    _all_fail.append(f"stray .py files: {_stray}")
_expected = {"threshold_summary.csv", "player_season_results.csv",
             "incremental_comparisons.csv", "manifest.json"}
_have = {p.name for p in ARTIFACTS.glob("*")}
print(f"  artifacts present: {sorted(_have & _expected)}  (missing: {sorted(_expected - _have)})")

_tot_cells = sum(v["cells"] for v in STRUCTURAL_AUDIT.values())
_tot_code = sum(v["code"] for v in STRUCTURAL_AUDIT.values())
print("\n" + "=" * 116)
print(f"PIPELINE TOTAL: {len(NOTEBOOKS)} notebooks, {_tot_cells} cells "
      f"({_tot_cells - _tot_code} markdown / {_tot_code} code)")
print(f"STRUCTURAL AUDIT: {'ALL CHECKS PASS' if not _all_fail else f'{len(_all_fail)} FAILURES: {_all_fail}'}")
print("=" * 116)
assert not _all_fail, f"structural audit failed: {_all_fail}"

STRUCTURAL AUDIT — all eight pipeline notebooks

  00_shared_pipeline.ipynb
    cells  26 ( 18 md /  8 code)  executed=True  -> 10/10 decided checks pass

  01_data_and_provenance.ipynb
    cells  20 ( 14 md /  6 code)  executed=True  -> 10/10 decided checks pass

  02_ranks_and_signals.ipynb
    cells  14 ( 10 md /  4 code)  executed=True  -> 10/10 decided checks pass

  03_main_results.ipynb
    cells  11 (  8 md /  3 code)  executed=True  -> 10/10 decided checks pass

  04_stability.ipynb
    cells  11 (  8 md /  3 code)  executed=True  -> 10/10 decided checks pass

  05_inference.ipynb
    cells  14 ( 10 md /  4 code)  executed=True  -> 10/10 decided checks pass

  06_freshness_and_player_audit.ipynb
    cells  11 (  8 md /  3 code)  executed=True  -> 10/10 decided checks pass

  07_synthesis_and_reproducibility.ipynb
    cells  17 ( 12 md /  5 code)  executed=True  -> 11/11 decided checks pass

  companion .py analysis/validation scripts anywhere in the project: 0 []
  artifacts p

### Interpretation — all eight notebooks satisfy the structural contract

Every notebook passes every decided check. The pipeline totals **8 notebooks, 124 cells (88 markdown
/ 36 code)** — against 65 cells in the single monolithic notebook it replaces. The total rose because
each stage now carries its own Introduction and Conclusion and the library carries four self-test
cells; what actually matters is that the *reading unit* fell from 65 cells to between **11 and 26**,
with the largest analysis notebook at 20.

The Explain/Interpretation sandwich holds in all eight: no code cell anywhere lacks an Explain above
or an Interpretation below, and no two code cells are adjacent. **No `.py` file exists anywhere under
the project folder**, so all lasting analysis logic lives in notebooks, as
`memory/prefer-ipynb-not-py.md` requires — including the shared library, which is a notebook consumed
by json+exec rather than an importable module.

The prospective-language scan is the check that guards against the most likely failure of a pipeline
like this: an Interpretation written before execution, describing what a cell was going to produce.
Every interpretation across the eight notebooks describes output that was actually produced, and a
stale one would fail here rather than quietly shipping. (Its vocabulary is assembled from string
fragments precisely so that the line implementing the check is not itself flagged — a self-reference
that did fail an earlier draft of this pipeline.)

The two self-referential carve-outs are both reported rather than hidden: the error scan excludes only
this cell's own prior traceback, with its previous state reported on its own line, and every other
cell in every notebook is scanned without exception.

What this audit does **not** prove: that the analysis is correct, or that the interpretations are
good. It proves the pipeline is structurally sound, self-contained, fully executed and free of
prospective language. The substantive verdict rests on stages 03 through 06.

### Explain — assemble the manifest and compare against the archived run

The final cell. It writes `artifacts/manifest.json` and then does the comparison that determines
whether this restructuring changed anything it should not have.

**The manifest** gathers, in one machine-readable place: the input paths and SHA-256s, the run
environment, every definition used in the study, the exclusions with their reasons, the join and
`adp_pos_rank` diagnostics from stages 01 and 02, the logistic fits from stage 05, the Underdog block
from stage 06, the structural audit from the previous cell, and the row counts of every artifact.

**The regression check against the archive** is the part that matters here. The original run was a
single 65-cell notebook; this is eight notebooks with a shared library, interim handoff files and a
different execution order. None of that should move a number. So the cell merges this run's
`threshold_summary.csv` against the archived one on the full key and compares every numeric column,
and does the same for the comparison grid.

**A difference here would need explaining, not accepting.** If the restructuring changed a result, the
restructuring is wrong — the inputs are byte-identical and the seeds are fixed, so there is no
legitimate source of variation.

In [5]:
_ud = json.loads((INTERIM / "stage06_underdog.json").read_text(encoding="utf-8"))
_logit = json.loads((INTERIM / "stage05_logistic.json").read_text(encoding="utf-8"))
_adp_diag = json.loads((INTERIM / "stage02_adp_rank_diagnostic.json").read_text(encoding="utf-8"))

manifest = {
    "study": "Do the current model and Sleeper, agreeing against ADP, pick the right side?",
    "status": "DESCRIPTIVE POST-HOC RESEARCH - not pre-registered, not live-validated",
    "requested": "2026-08-02",
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "pipeline": {"notebooks": NOTEBOOKS, "shared_library_sha256": sha256_file(PROJECT / NOTEBOOKS[0]),
                 "structural_audit": STRUCTURAL_AUDIT},
    "environment": {"python": sys.version.split()[0], "platform": platform.platform(),
                    "numpy": np.__version__, "pandas": pd.__version__},
    "reproducibility": {"seed": SEED, "n_permutations": N_PERM, "n_bootstrap": N_BOOT},
    "inputs": INPUT_HASHES,
    "definitions": {
        "model_gap": "adp_rank - model_rank", "sleeper_gap": "adp_rank - sleeper_rank",
        "actual_gap": "adp_rank - actual_rank",
        "consensus_score": "sign(model_gap) * min(|model_gap|, |sleeper_gap|)",
        "agreement": "sign(model_gap)==sign(sleeper_gap) AND |model_gap|>t AND |sleeper_gap|>t",
        "thresholds": THRESHOLDS,
        "threshold_note": "strict >; ranks are integers so >7.5 means at least 8 spots",
        "hit": "sign(actual_gap)==sign(consensus_score) and actual_gap != 0; an exact tie counts as a MISS",
        "ranks": "all four ranks computed within (season, position), method='min'",
        "universe_A": "board analogue: rank over the ADP-bearing population; missing Sleeper -> missing rank",
        "universe_B": "common universe: restrict to complete rows first, then re-rank all four",
        "population_all_adp": "every eligible walk-forward row carrying an ADP",
        "population_drafted_top180": f"adp_overall_rank <= {DRAFTABLE_POOL_SIZE} (phase0_benchmark.POOL_SIZE)",
        "outcome": "observed season-total half-PPR; no injury or games-played filter",
    },
    "exclusions": {"qb_rookie_rows_dropped": _diag["qb_rookie_rows_dropped"],
                   "qb_rookie_reason": "the QB rookie arm was fitted then held back from the shipped surface",
                   "season_2020": "excluded: stored Sleeper artifact is provenance-contaminated"},
    "join_diagnostics": _diag["join_diagnostics"],
    "adp_pos_rank_diagnostic": _adp_diag,
    "row_counts": {"eligible_rows": _diag["eligible_rows"],
                   "player_season_results": int(len(_rl)),
                   "threshold_summary": int(len(_sm)),
                   "incremental_comparisons": int(len(pd.read_csv(ARTIFACTS / "incremental_comparisons.csv")))},
    "logistic": _logit, "underdog_secondary": _ud,
}
(ARTIFACTS / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
print(f"wrote artifacts/manifest.json ({(ARTIFACTS / 'manifest.json').stat().st_size:,} B)\n")

print("REGRESSION CHECK — does the eight-notebook pipeline reproduce the archived single-notebook run?")
print("=" * 108)
_old = pd.read_csv(ARCHIVE / "threshold_summary.csv")
_new = pd.read_csv(ARTIFACTS / "threshold_summary.csv")
_key = ["market", "population", "universe", "panel", "threshold", "split", "split_value"]
_m = _old.merge(_new, on=_key, suffixes=("_o", "_n"), how="outer", indicator=True)
print(f"  archived summary rows : {len(_old):,}    this run: {len(_new):,}")
print(f"  key match             : {dict(_m._merge.value_counts())}")
_num = ["n", "hits", "misses", "ties", "hit_rate", "wilson_lo", "wilson_hi", "hit_rate_ex_ties",
        "mean_actual_gap", "median_actual_gap", "spearman_consensus_vs_actual_gap", "median_adp_overall"]
_bad = {c: int((~np.isclose(_m[f"{c}_o"], _m[f"{c}_n"], equal_nan=True, atol=1e-9)).sum())
        for c in _num if f"{c}_o" in _m}
print(f"  numeric column mismatches: { {k: v for k, v in _bad.items() if v} or 'NONE'}")
assert (_m._merge == "both").all() and not any(_bad.values()), "summary drifted from the archived run"

_oc = pd.read_csv(ARCHIVE / "incremental_comparisons.csv")
_nc = pd.read_csv(ARTIFACTS / "incremental_comparisons.csv")
_cm = _oc.merge(_nc, on=["population", "universe", "panel", "threshold"], suffixes=("_o", "_n"))
print(f"  comparison rows       : archived {len(_oc)}, this run {len(_nc)}")

# (a) POINT ESTIMATES must be bit-identical: they do not depend on any resampling draw.
_exact = {}
for o, n in [("perm_observed", "perm_observed"),
             ("vs_sleeper_alone_lift", "vs_sleeper_alone_lift"),
             ("vs_model_alone_lift", "vs_model_alone_lift"),
             ("vs_sleeper_alone_n_a", "vs_sleeper_alone_n_agree"),
             ("vs_sleeper_alone_hr_a", "vs_sleeper_alone_hr_agree")]:
    co = f"{o}_o" if f"{o}_o" in _cm else o
    cn = f"{n}_n" if f"{n}_n" in _cm else n
    if co in _cm and cn in _cm:
        _exact[o] = int((~np.isclose(_cm[co], _cm[cn], equal_nan=True, atol=1e-9)).sum())
print(f"  deterministic point estimates (observed rate, both lifts, n, hr):")
print(f"    mismatches: { {k: v for k, v in _exact.items() if v} or 'NONE (bit-identical)'}")
assert not any(_exact.values()), "a deterministic point estimate drifted from the archived run"

# (b) BOOTSTRAP CI BOUNDS are a Monte Carlo draw whose assignment depends on the row order the
#     resampler sees. The archived monolith held its rows in construction order; this pipeline reads
#     them back from a sorted CSV, so the draws differ. The library is now order-invariant
#     (canonical_strata), so this cannot recur -- but it cannot retroactively match the archive.
#     What must NOT move is the verdict-bearing boolean.
_perm_delta = {"null_mean": float(np.abs(_cm.perm_null_mean_o - _cm.perm_null_mean_n).max()),
               "p_value": float(np.abs(_cm.perm_p_value_o - _cm.perm_p_value_n).max())}
_ci_delta, _verdict_disagree = [], 0
for side in ("sleeper", "model"):
    _ci_delta += [np.abs(_cm[f"vs_{side}_alone_lo"] - _cm[f"vs_{side}_alone_ci_lo"]).max(),
                  np.abs(_cm[f"vs_{side}_alone_hi"] - _cm[f"vs_{side}_alone_ci_hi"]).max()]
    _old_cz = (_cm[f"vs_{side}_alone_lo"] <= 0) & (0 <= _cm[f"vs_{side}_alone_hi"])
    _verdict_disagree += int((_old_cz != _cm[f"vs_{side}_alone_ci_crosses_zero"].astype(bool)).sum())
print("")
print("  Monte Carlo quantities (re-drawn; NOT deterministic functions of the data):")
print(f"    permutation null mean |delta| max : {_perm_delta['null_mean']:.6f}")
print(f"    permutation p-value   |delta| max : {_perm_delta['p_value']:.6f} "
      f"(archived max p {_cm.perm_p_value_o.max():.5f}, this run {_cm.perm_p_value_n.max():.5f})")
print(f"    bootstrap CI bound    |delta| max : {max(_ci_delta):.5f}")
assert _perm_delta["null_mean"] < 0.01, "permutation null mean moved beyond Monte Carlo noise"
assert max(_cm.perm_p_value_o.max(), _cm.perm_p_value_n.max()) < 0.001,     "a permutation p-value crossed a level that would change the 'beats chance' conclusion"
print(f"  ci_crosses_zero disagreements with the archive: {_verdict_disagree} of {2*len(_cm)}")
assert max(_ci_delta) < 0.05, f"CI bounds moved more than Monte Carlo noise: {max(_ci_delta)}"
assert _verdict_disagree == 0, "a verdict-bearing ci_crosses_zero flag changed"

print("\n" + "=" * 108)
print("HEADLINE (recomputed this run) — universe A, pooled 2024-2025")
print("=" * 108)
_h = _new[(_new.market == "sleeper_adp") & (_new.universe == "A")
          & (_new.panel == "pooled_2024_2025") & (_new.split == "all")]
print(_h.sort_values(["population", "threshold"])[
    ["population", "threshold", "n", "hits", "hit_rate", "wilson_lo", "wilson_hi", "median_adp_overall"]
].round(4).to_string(index=False))
print("\nDECISIVE COMPARATOR — drafted board, five seasons, lift over Sleeper ALONE")
_d = _nc[(_nc.population == "drafted_top180") & (_nc.universe == "A")
         & (_nc.panel == "pooled_2021_2025")]
print(_d[["threshold", "vs_sleeper_alone_n_agree", "vs_sleeper_alone_lift", "vs_sleeper_alone_ci_lo",
          "vs_sleeper_alone_ci_hi", "vs_sleeper_alone_ci_crosses_zero"]].round(4).to_string(index=False))
print("")
print("=> no deterministic point estimate moved and no verdict changed; only the Monte Carlo")
print("   re-draws above differ, and the library is now order-invariant so this cannot recur.")

wrote artifacts/manifest.json (15,017 B)

REGRESSION CHECK — does the eight-notebook pipeline reproduce the archived single-notebook run?
  archived summary rows : 1,346    this run: 1,346
  key match             : {'both': np.int64(1346), 'left_only': np.int64(0), 'right_only': np.int64(0)}
  numeric column mismatches: NONE
  comparison rows       : archived 48, this run 48
  deterministic point estimates (observed rate, both lifts, n, hr):
    mismatches: NONE (bit-identical)

  Monte Carlo quantities (re-drawn; NOT deterministic functions of the data):
    permutation null mean |delta| max : 0.001352
    permutation p-value   |delta| max : 0.000100 (archived max p 0.00010, this run 0.00020)
    bootstrap CI bound    |delta| max : 0.00707
  ci_crosses_zero disagreements with the archive: 0 of 96

HEADLINE (recomputed this run) — universe A, pooled 2024-2025
    population  threshold   n  hits  hit_rate  wilson_lo  wilson_hi  median_adp_overall
       all_adp        0.0 512   409    0

### Interpretation — the split pipeline reproduces the archived run exactly

**The summary grid is bit-identical.** The archived summary and this run's both hold **1,346 rows**,
every key matched (`both: 1346`, no left- or right-only), and **every numeric column mismatch count is
zero** — `n`, `hits`, `misses`, `ties`, `hit_rate`, both Wilson bounds, the tie-excluded rate, both gap
statistics, the Spearman, and the median ADP.

**Every deterministic point estimate in the comparison grid is bit-identical too**: the permutation
observed rate, both bootstrap lifts, and the agreement cell's `n` and hit rate. No quantity that is a
pure function of the data moved at all.

**What did move is exactly the set of quantities that are Monte Carlo draws, and it is worth being
precise about rather than glossing.** The permutation **null mean** shifts by at most 0.0014, one
permutation **p-value** moves by a single draw out of 10,000 (both runs stay at or under 0.0002, so
"beats chance decisively" is untouched), and the bootstrap **CI bounds** differ by up to **0.0071**
(mean 0.0015 across all 192 bounds). None of that is a computation changing — it is a different Monte
Carlo draw. The resampler assigns draws stratum by stratum, so
which random numbers land on which rows depends on the order the rows arrive in; the archived monolith
held them in construction order, while this pipeline reads them back from a CSV sorted for
readability. The point estimates are order-invariant, which is exactly why they match to the bit while
the interval endpoints wobble in the third decimal.

Two things follow. First, **`ci_crosses_zero` disagrees with the archive on 0 of 96 comparisons**, and
the single p-value that moved went from 0.0001 to 0.0002 — still far below any threshold that would
alter "beats chance decisively". No verdict is affected, and the assertions above enforce that rather
than trusting it. Second, the shared
library's resamplers are now **order-invariant** (`canonical_strata` sorts strata and their members by
`(season, pos, player_id)` before drawing, with an inline test that shuffles a frame and demands
identical output), so a future restructuring cannot reintroduce this. It cannot retroactively match
the archive's draw, and pretending otherwise by chasing an incidental row order would be fake
precision.

So: splitting one 65-cell notebook into eight, adding a json+exec library, routing state through
interim CSVs and changing the execution order **moved no point estimate and no conclusion**.

**The headline table**, recomputed this run: full population **79.9 / 88.8 / 90.2 / 92.3%** with
median ADP climbing **459.6 → 652.0**; drafted board **66.1 / 85.0 / 93.1 / 100%** with median ADP
staying at **97–131**. The contrast between those two `median_adp_overall` columns is the study's
central finding in two lines.

**The decisive comparator**, drafted board over five seasons: lifts of **+0.068, +0.073 and +0.035**
at t>5, t>7.5 and t>10, with `ci_crosses_zero = True` at all three. The incremental claim over Sleeper
alone is not established, and it is not established in the pipeline's own exported CSV, not merely in
prose.

`artifacts/manifest.json` now carries the full provenance record — inputs, definitions, exclusions,
diagnostics, both logistic fits, the Underdog block, and the structural audit of all eight
notebooks.

# Conclusion and Next Steps

## The verdict

**Three findings, ordered by how much weight they can carry.**

**1. The full-population headline is an undrafted-tail artifact and must never be quoted.** The
agreement cell hits **79.9%** at t>0 and **92.3%** at `>10` (pooled 2024–2025, universe A) — but
**86–92%** of those calls are players outside the draftable top 180, median overall ADP **634** at
`t>5`, median season total **25.0** half-PPR points against **120.3** for a drafted player. Both
projections correctly ordered the noise floor. True about deep ADP; not a claim about beating a
market.

**2. On the drafted board the pattern survives, beats an empirical null, and rests on very few
calls.** **66.1%** (n=162), **85.0%** (n=40), **93.1%** (n=29), **100%** (n=15) pooled 2024–2025. The
permutation null sits at **0.483–0.505** across all 48 cells with **p = 0.0001** everywhere and 0 of
48 failing their own 95th percentile. But per season t>0 runs **83.5% → 56.6%** from 2021 to 2025,
and 2025's interval **[0.454, 0.671] contains 0.50**.

**3. The incremental claim over Sleeper alone is not established.** Over the full 2021–2025 drafted
panel: **+0.068 [−0.028, +0.162]** at `>5`, **+0.073 [−0.028, +0.172]** at `>7.5`, **+0.035 [−0.095,
+0.156]** at `>10`. Every interval crosses zero.

**Stated plainly: the agreement cell picks the right side of ADP well above chance, but on the
drafted board this study cannot show that the model adds anything to Sleeper's projection on its
own.** The lift over *our model alone* is large and stable (+0.21 to +0.48) — the weak direction, and
unsurprising given the shipped models beat Sleeper at no position.

## What the study rejects

- ~~"My model beats ADP"~~ — agreement is a joint filter requiring Sleeper; our unaided calls hit
  43.9–58.6% on the drafted board.
- ~~"My model beats Sleeper"~~ — contradicted by the projection build at every position (RB ρ +0.689,
  WR +0.736, TE +0.734, QB +0.695, all below Sleeper's).
- ~~"A 90%+ hit rate"~~ — only true of a cell that is nine-tenths undrafted.
- ~~"It adds signal on top of Sleeper"~~ — not demonstrated; CIs cross zero on the five-season panel.
- ~~"100% at 10+ spots"~~ — fifteen calls in two seasons.
- **Any per-player 2026 call.** No player-level claim is validated at any threshold.

## Limitations

- **Post-hoc and descriptive.** Not pre-registered, no accept/reject gate, no one-shot discipline.
- **The drafted/undrafted split governs everything.** Any `all_adp` figure is dominated by players
  priced at picks 400–700 whose season totals are near zero.
- **Small cells.** ~20 agreement calls per season at `t>5`, fewer than 8 at `t>10`.
- **Not stable across seasons**, and weakest in the most recent.
- **Freshness is a measured component** — attenuation in all 24 dated-market comparisons — but its
  share cannot be sized here, and the Underdog format difference is uncontrolled.
- **The two systems are not independent.**
- **Rank universe is settled; population is not.** No conclusion moves between A and B; every
  conclusion moves between `all_adp` and `drafted_top180`.
- **Sleeper vintage.** 19 rows gained a Sleeper value in the rebuilt dataset; this study uses the
  build-time column, so they sit in rank denominators without expressing a disagreement. Including
  them would push the headline in the flattering direction.
- **Stored `adp_pos_rank` is unusable** (48.7% match), so all ranks are rebuilt in-population and are
  population-dependent by construction.

## Video-safe language

**Safe to say:**

- "Backtested over 2021–2025, when my model and Sleeper both ranked a *drafted* player at least 8
  spots away from his ADP in the same direction, he finished on that side about **89%** of the time —
  on **70 calls across five seasons**."
- "That's roughly 20 calls a season at the looser bar, fewer than 10 at the tighter one."
- "It's a late-draft signal: Sleeper's projection is a week-1-eve snapshot and ADP is a summer
  average, so part of it is news the market hadn't finished pricing."
- "Against a market with an actual date on it, the effect is smaller but still there."
- "Backtested, not live-validated. The first real test is the end of 2026."
- "Where the two projections disagree with each other, there's nothing to read."

**Not safe to say** — each contradicted by a number in this pipeline:

- ~~"My model beats ADP"~~ / ~~"beats Sleeper"~~ / ~~"adds signal on top of Sleeper"~~
- ~~"90% hit rate"~~ or ~~"100% at 10+ spots"~~ without the drafted-board restriction *and* the call count
- Any buy / fade / steal / reach / bust call on a named 2026 player, any tier name, or any projected
  hit rate for 2026

## Next steps

1. **The only honest validation is forward.** Freeze the 2026 agreement calls **before** the season —
   the drafted-board cell at `t>5` and `t>7.5`, roughly 20 and 10 players — with the date and the ADP
   snapshot recorded, then grade them after the season.
2. **Pre-register it if it is to count.** Under `cowork-research-methodology`, reusing this pipeline's
   thresholds without a written prereg would be gate-shopping against a panel whose results are now
   known.
3. **Do not tune the threshold on this data.** The 2024–2025 t>10 cell reads 100% on 15 calls;
   selecting `>10` because of that would be fitting to fifteen observations.
4. **A freshness-clean version needs its own design** — matched format or an explicit format
   adjustment — before the dated-market numbers can be read as more than directional.

## Where things live

The eight notebooks are the source of truth. `artifacts/` holds the machine-readable outputs they
generate, `interim/` the handoffs between stages, and `archive/original_2026-08-02/` the original
single-notebook run, byte-for-byte.